In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_val_predict, cross_validate
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer

### Check the performance of Random Forests with different training samples

In [ ]:
# Set seed for reproducibility and test_size = 0.2
seed = 25
np.random.seed(seed)
random.seed(seed)

# Use optimal hyperparameters found by 5-fold cross-validation for the baseline RF model
origin_best_params = {'bootstrap': False, 'criterion': 'squared_error', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}

### Baseline RF model

In [ ]:
# Read data
original_data = gpd.read_file(r"original_training_data.gpkg")
print(original_data.shape)

# Define features and target
X_origin = original_data.drop(columns=['soc', 'source', 'geometry'])
X_origin = pd.get_dummies(X_origin)
y_origin = original_data['soc']

origin_model = RandomForestRegressor(**origin_best_params, random_state=seed)

# 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=seed)

y_cv_pred = cross_val_predict(
    origin_model,
    X_origin,       
    y_origin,
    cv=cv,
    n_jobs=-1
)

# CV performance metrics
cv_r2 = r2_score(y_origin, y_cv_pred)
cv_rmse = np.sqrt(mean_squared_error(y_origin, y_cv_pred))
cv_mae = mean_absolute_error(y_origin, y_cv_pred)

print(f"CV R2   : {cv_r2:.2f}")
print(f"CV RMSE : {cv_rmse:.2f}")
print(f"CV MAE  : {cv_mae:.2f}")

### Updated RF model

In [ ]:
# Load the combined dataset (1004 samples)
data_combined = gpd.read_file(r"updated_training_data.gpkg")
print(data_combined.shape)

# Define features and target
X_new = data_combined.drop(columns=['soc', 'origin', 'geometry'])
X_new = pd.get_dummies(X_new)
y_new = data_combined['soc']

updated_model = RandomForestRegressor(**origin_best_params, random_state=seed)

# 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=seed)

y_cv_pred = cross_val_predict(
    updated_model,
    X_new,       
    y_new,
    cv=cv,
    n_jobs=-1
)

# CV performance metrics
cv_r2 = r2_score(y_new, y_cv_pred)
cv_rmse = np.sqrt(mean_squared_error(y_new, y_cv_pred))
cv_mae = mean_absolute_error(y_new, y_cv_pred)

print(f"CV R2   : {cv_r2:.2f}")
print(f"CV RMSE : {cv_rmse:.2f}")
print(f"CV MAE  : {cv_mae:.2f}")